In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!git clone https://github.com/RIA-lab/MMTD.git

In [ ]:
%cd /kaggle/working/MMTD

In [ ]:
!pip install -r requirements.txt


In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BertConfig, BeitForImageClassification, BeitConfig
from transformers.modeling_outputs import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class PhishGuard_MMTD(torch.nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(PhishGuard_MMTD, self).__init__()
        
        # 1. Temel Modeller
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        # 2. VRAM TASARRUFU: Ana gövdeleri dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        # 3. ASİMETRİK CO-ATTENTION (Parametre yarıya düştü!)
        self.asymmetric_attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # 4. DİNAMİK GATING KATMANI
        self.gating_layer = nn.Linear(768 * 2, 768)

        # 5. SINIFLANDIRICI (Classifier)
        self.classifier = nn.Linear(768, 2) 
        self.num_labels = 2
        
        # Modality Dropout Oranı
        self.modality_dropout_prob = 0.1 

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None):
        # A. Modellerden Vektörleri Al
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_hidden = text_outputs.hidden_states[12]
        image_hidden = image_outputs.hidden_states[12]
        
        # B. MODALITY DROPOUT (Eğitim sırasında ezberi engeller)
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < self.modality_dropout_prob:
                image_hidden = torch.zeros_like(image_hidden) # Resmi karart
            elif rand_val < (self.modality_dropout_prob * 2):
                text_hidden = torch.zeros_like(text_hidden)   # Metni karart
        
        # C. ASİMETRİK CO-ATTENTION
        # Metin (Query) -> Resim (Key, Value)
        text_attended_by_image, _ = self.asymmetric_attn(query=text_hidden, key=image_hidden, value=image_hidden)
        
        # D. HAVUZLAMA
        text_pooled = text_hidden[:, 0, :]
        text_attended_pooled = text_attended_by_image[:, 0, :]
        
        # E. DİNAMİK GATING
        concat_features = torch.cat([text_pooled, text_attended_pooled], dim=1)
        gate_weights = torch.sigmoid(self.gating_layer(concat_features))
        
        fused_features = gate_weights * text_pooled + (1 - gate_weights) * text_attended_pooled
        
        # F. SINIFLANDIRMA
        logits = self.classifier(fused_features)
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# Email_dataset.py dosyasındaki sürüm uyumsuzluğunu çözen yama
file_path = '/kaggle/working/MMTD/Email_dataset.py'

with open(file_path, 'r') as file:
    file_data = file.read()

# Eski ismi yeni isimle değiştir
file_data = file_data.replace('BeitFeatureExtractor', 'BeitImageProcessor')

# Düzeltilmiş kodu dosyaya geri yaz
with open(file_path, 'w') as file:
    file.write(file_data)

print("✅ Yama uygulandı! BeitFeatureExtractor başarıyla BeitImageProcessor olarak değiştirildi.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# Email_dataset.py dosyasındaki CLIP sürüm uyumsuzluğunu çözen yama
file_path = '/kaggle/working/MMTD/Email_dataset.py'

with open(file_path, 'r') as file:
    file_data = file.read()

# Eski ismi yeni isimle değiştir
file_data = file_data.replace('CLIPFeatureExtractor', 'CLIPImageProcessor')

# Düzeltilmiş kodu dosyaya geri yaz
with open(file_path, 'w') as file:
    file.write(file_data)

print("✅ Yama 2 uygulandı! CLIPFeatureExtractor başarıyla CLIPImageProcessor olarak değiştirildi.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os
import glob

# Projedeki tüm Python dosyalarını bul
proje_dizini = '/kaggle/working/MMTD/**/*.py'
python_dosyalari = glob.glob(proje_dizini, recursive=True)

degisen_dosya_sayisi = 0

for dosya_yolu in python_dosyalari:
    with open(dosya_yolu, 'r', encoding='utf-8') as dosya:
        icerik = dosya.read()
    
    # Eğer dosyada 'FeatureExtractor' kelimesi geçiyorsa değiştir
    if 'FeatureExtractor' in icerik:
        yeni_icerik = icerik.replace('FeatureExtractor', 'ImageProcessor')
        
        with open(dosya_yolu, 'w', encoding='utf-8') as dosya:
            dosya.write(yeni_icerik)
            
        print(f"✅ Temizlendi: {dosya_yolu.split('/')[-1]}")
        degisen_dosya_sayisi += 1

print(f"\n🚀 OPERASYON TAMAM! Toplam {degisen_dosya_sayisi} dosyada tüm isimler tek seferde güncellendi.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# Eksik olan torchtext ve onun yardımcı kütüphanesi portalocker'ı kuruyoruz
!pip install torchtext portalocker


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# utils.py içindeki sistemi çökerten gereksiz GloVe importunu silen yama
file_path = '/kaggle/working/MMTD/utils.py'

with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()

with open(file_path, 'w', encoding='utf-8') as file:
    for line in lines:
        # Eğer satırda torchtext geçmiyorsa dosyaya geri yaz (Yani torchtext'i çöpe at)
        if 'torchtext' not in line:
            file.write(line)

print("✅ Ameliyat başarılı! Gereksiz torchtext bağımlılığı koddan tamamen söküldü.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os
import glob

# Projedeki tüm Python dosyalarını bul
proje_dizini = '/kaggle/working/MMTD/**/*.py'
python_dosyalari = glob.glob(proje_dizini, recursive=True)

temizlenen_dosya_sayisi = 0

for dosya_yolu in python_dosyalari:
    with open(dosya_yolu, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        
    yazilacak_satirlar = []
    degisiklik_var_mi = False
    
    for line in lines:
        if 'torchtext' not in line:
            yazilacak_satirlar.append(line)
        else:
            degisiklik_var_mi = True
            
    # Sadece değişiklik yaptıysak dosyaya geri yaz (diski yormayalım)
    if degisiklik_var_mi:
        with open(dosya_yolu, 'w', encoding='utf-8') as file:
            file.writelines(yazilacak_satirlar)
        print(f"🧹 Temizlendi: {dosya_yolu.split('/')[-1]}")
        temizlenen_dosya_sayisi += 1

print(f"\n✅ OPERASYON TAMAM! Toplam {temizlenen_dosya_sayisi} dosyadan torchtext kalıntıları tamamen silindi.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# models.py içindeki sınıf adını main.py'nin beklediği isme (MMTD) çeviren yama
file_path = '/kaggle/working/MMTD/models.py'

with open(file_path, 'r', encoding='utf-8') as f:
    icerik = f.read()

# Sınıf adlarını değiştir
icerik = icerik.replace('class PhishGuard_MMTD', 'class MMTD')
icerik = icerik.replace('super(PhishGuard_MMTD', 'super(MMTD')

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(icerik)

print("✅ İsim etiketi düzeltildi! Yeni Asimetrik motorumuz artık sistemin tanıdığı 'MMTD' adını taşıyor.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os

# 1. Veri setinin Kaggle'daki gerçek yerini (Input klasörünü) otomatik bul
arama = os.popen('find /kaggle/input -name EDP.csv').read().strip().split('\n')

if arama and arama[0]:
    edp_yolu = arama[0]
    kaynak_klasor = os.path.dirname(edp_yolu) # İçinde EDP.csv ve resimlerin olduğu asıl klasör
    
    print(f"✅ Veri Seti Bulundu: {kaynak_klasor}")
    
    # 2. Kodun aradığı 'DATA' klasörünün iskeletini oluştur
    os.makedirs('/kaggle/working/MMTD/DATA', exist_ok=True)
    
    # 3. Eski bir kalıntı varsa temizle
    os.system('rm -rf /kaggle/working/MMTD/DATA/email_data')
    
    # 4. Sihirli Sembolik Link (Kısayol) - Veriyi kopyalamadan anında bağlar
    os.system(f'ln -s "{kaynak_klasor}" /kaggle/working/MMTD/DATA/email_data')
    
    print("🔗 Veri köprüsü başarıyla kuruldu! Artık kod veriyi bulabilecek.")
else:
    print("❌ DİKKAT: Kaggle Input klasöründe EDP.csv bulunamadı. Sağ üstteki 'Add Data' butonundan veri setini eklediğine emin misin?")

In [ ]:
import os
import glob

# 1. Proje ana dizininde olduğumuzdan emin olalım
%cd /kaggle/working/MMTD

# 2. README'de istenen 'DATA' klasörünü oluştur ve içine gir
os.makedirs('DATA', exist_ok=True)
%cd DATA

# 3. Google Drive indirme aracını kur
!pip install gdown -q

# 4. Görseldeki Drive linkinden veriyi doğrudan Kaggle'a çek (Orijinal ID)
print("📥 Veri seti Google Drive'dan indiriliyor (Kaggle'ın hızıyla çok kısa sürecek)...")
!gdown --id 1w4HxNf099lQ41mhMyXzbGwl429qBGvbY

# 5. İnen zip dosyasını bul ve buraya çıkart
zip_dosyalari = glob.glob('*.zip')
if zip_dosyalari:
    hedef_zip = zip_dosyalari[0]
    print(f"📦 {hedef_zip} bulundu, zipten çıkarılıyor...")
    !unzip -q {hedef_zip}
    print("✅ Zipten çıkarma işlemi tamamlandı!")
else:
    print("❌ Zip dosyası bulunamadı. İndirme işleminde bir hata olmuş olabilir.")

# 6. Tekrar ana dizine dön
%cd /kaggle/working/MMTD
print("🚀 YAKIT DEPOYA DOLDURULDU! Eğitime hazırız.")

In [ ]:
!gdown --id "1w4HxNf099lQ41mhMyXzbGwI429qBGvbY"

In [ ]:
# DATA klasörünü oluştur ve indirilen zip dosyasını oraya çıkart
!mkdir -p DATA
!unzip -q email_data.zip -d DATA/

print("✅ Veri seti başarıyla zipten çıkarıldı ve yerine yerleştirildi!")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# 1. main.py Klasör Hatası Bypass Yaması
main_path = '/kaggle/working/MMTD/main.py'
with open(main_path, 'r', encoding='utf-8') as f:
    main_code = f.read()

# Olmayan klasörleri atla, sahte bir döngü yarat
main_code = main_code.replace('os.listdir(bert_checkpoint_path)', '["fold_1"]')
main_code = main_code.replace('os.listdir(beit_checkpoint_path)', '["fold_1"]')

with open(main_path, 'w', encoding='utf-8') as f:
    f.write(main_code)

# 2. models.py HuggingFace Doğrudan İndirme Yaması
models_path = '/kaggle/working/MMTD/models.py'
with open(models_path, 'r', encoding='utf-8') as f:
    models_code = f.read()

# Değişkenleri yoksay, doğrudan orijinal modellerin ismini yazdır
models_code = models_code.replace('BertForSequenceClassification.from_pretrained(bert_pretrain_weight)', 'BertForSequenceClassification.from_pretrained("bert-base-uncased")')
models_code = models_code.replace('BeitForImageClassification.from_pretrained(beit_pretrain_weight)', 'BeitForImageClassification.from_pretrained("microsoft/beit-base-patch16-224-pt22k")')

with open(models_path, 'w', encoding='utf-8') as f:
    f.write(models_code)

print("✅ SİSTEM HACKLENDİ: Model artık yerel klasör aramayacak, doğrudan HuggingFace'ten taptaze BERT ve BEiT indirecek!")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# main.py'yi kandırmak için sahte (içi boş) klasörleri ve dosyaları oluşturuyoruz
!mkdir -p ./output/bert/checkpoints/fold_1
!mkdir -p ./output/beit/checkpoints/fold_1

# İçlerine sahte model ve config dosyaları koyalım (0 byte boyutunda)
!touch ./output/bert/checkpoints/fold_1/pytorch_model.bin
!touch ./output/bert/checkpoints/fold_1/config.json

!touch ./output/beit/checkpoints/fold_1/pytorch_model.bin
!touch ./output/beit/checkpoints/fold_1/config.json

print("✅ KUKLA DOSYALAR YERLEŞTİRİLDİ! main.py artık bu sahte dosyaları bulup mutlu olacak.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os

# main.py'nin arayabileceği tüm olası eski modellerin listesi
eski_modeller = ['bert', 'beit', 'dit', 'clip', 'vilt']

for model in eski_modeller:
    klasor_yolu = f'./output/{model}/checkpoints/fold_1'
    os.makedirs(klasor_yolu, exist_ok=True)
    
    # İçlerine sahte model ve config dosyaları koyalım
    open(f'{klasor_yolu}/pytorch_model.bin', 'w').close()
    open(f'{klasor_yolu}/config.json', 'w').close()

print("✅ GLOBAL KUKLA OPERASYONU TAMAM! Tüm eski hayalet klasörler sahte dosyalarla dolduruldu.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os
# wandb'yi kökten susturan en güçlü sistem komutu
os.environ["WANDB_MODE"] = "disabled"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# main.py içindeki sürüm hatasını (evaluation_strategy -> eval_strategy) düzelten yama
file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as file:
    file_data = file.read()

# Eski argüman ismini yenisiyle değiştir
file_data = file_data.replace('evaluation_strategy', 'eval_strategy')

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(file_data)

print("✅ Yama uygulandı! 'evaluation_strategy' başarıyla 'eval_strategy' olarak güncellendi.")

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
# main.py içindeki TrainingArguments hatalarını gideren toplu yama
file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as file:
    file_data = file.read()

# 1. Strateji ismini güncelle
file_data = file_data.replace('evaluation_strategy', 'eval_strategy')

# 2. Hata veren 'overwrite_output_dir' argümanını devre dışı bırak 
# (Zaten biz elle klasör temizliği yaptığımız için buna ihtiyacımız yok)
file_data = file_data.replace('overwrite_output_dir = True', 'overwrite_output_dir = False')
file_data = file_data.replace('overwrite_output_dir=True', 'overwrite_output_dir=False')

with open(file_path, 'w', encoding='utf-8') as file:
    file.write(file_data)

print("✅ Yama uygulandı! main.py artık modern TrainingArguments ile uyumlu.")

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
import os

file_path = '/kaggle/working/MMTD/main.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Hata çıkaran eski TrainingArguments bloğunu tamamen hedef alıyoruz.
# eval_strategy ve diğer güncel parametrelerle tertemiz bir blok yazıyoruz.

old_block_start = "args = TrainingArguments("
# Bu kısım genellikle ')' ile biter. O bloğu bulup değiştireceğiz.

new_args_block = """args = TrainingArguments(
    output_dir=args.output_dir,
    eval_strategy="epoch",  # Modern isim
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False, # Multimodal modellerde bu kritiktir!
    push_to_hub=False,
    report_to="none" # WandB sormasın diye garantili çözüm
)"""

# Basit bir replace yerine, daha güvenli bir yöntemle o kısmı yamalayalım
import re
# args = TrainingArguments(...) bloğunu yakalayan regex (parantez içini de alır)
content = re.sub(r"args = TrainingArguments\(.*?\)", new_args_block, content, flags=re.DOTALL)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ KRİTİK YAMA TAMAMLANDI: TrainingArguments modern sürüme göre baştan yazıldı.")

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!python main.py --output_dir ./output/PhishGuard_Asymmetric

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD # Bizim asimetrik modelimiz
from utils import SplitData

# 1. PARAMETRELER
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA
# Fold 1 üzerinden hızlıca eğitimi başlatıyoruz
split_data = SplitData('DATA/email_data/EDP.csv', 1)
train_df, val_df = split_data.train_df, split_data.test_df

train_dataset = EDPDataset(train_df, 'DATA/email_data/images')
val_dataset = EDPDataset(val_df, 'DATA/email_data/images')
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# Modellerimiz doğrudan HuggingFace'ten gelecek şekilde ayarlandı
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Versiyon)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir=os.path.join(output_dir, 'logs'),
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none"
)

# 5. TRAINER BAŞLAT
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("🚀 EĞİTİM BAŞLIYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ EĞİTİM TAMAMLANDI! Model {output_dir} klasörüne kaydedildi.")

In [ ]:
# Artık dosyanın içinde her şey tanımlı olduğu için ekstra argümana bile gerek yok
!python /kaggle/working/MMTD/main.py

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData

# 1. PARAMETRELER VE ORTAM
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA
# Not: Orijinal repo 'train' ve 'test' isimlerini kullanıyor.
split_data = SplitData('DATA/email_data/EDP.csv', 1)
train_df, val_df = split_data.train, split_data.test  # DÜZELTME BURADA

train_dataset = EDPDataset(train_df, 'DATA/email_data/images')
val_dataset = EDPDataset(val_df, 'DATA/email_data/images')
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# Bu kısım zaten HuggingFace'ten çekecek şekilde modellerini hazırlamıştık.
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Kaggle/Transformers Uyumu)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir=os.path.join(output_dir, 'logs'),
    logging_steps=10,
    remove_unused_columns=False, # Multimodal için kritik: Sütunları silme
    report_to="none"
)

# 5. TRAINER (EĞİTMEN) KURULUMU
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("🚀 TÜM ENGELLER AŞILDI, EĞİTİM BAŞLIYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ ZAFER! Model {output_dir} klasörüne kaydedildi.")

In [ ]:
!python /kaggle/working/MMTD/main.py

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
import pandas as pd
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData, metrics

# 1. ORTAM AYARLARI
output_dir = './output/PhishGuard_Asymmetric'
os.makedirs(output_dir, exist_ok=True)
os.environ["WANDB_MODE"] = "disabled"

# 2. VERİ SETİNİ HAZIRLA (Orijinal Callable Mantığı)
# Fold 1 için veriyi bölüyoruz
split_data = SplitData('DATA/email_data/EDP.csv', 5) # Orijinaldeki gibi 5 fold hazırlığı
train_df, test_df = split_data() # İŞTE KRİTİK DÜZELTME: Nesneyi fonksiyon gibi çağırıyoruz

# Orijinal kodunda path önce, df sonra geliyor: EDPDataset('path', df)
train_dataset = EDPDataset('DATA/email_data/pics', train_df)
test_dataset = EDPDataset('DATA/email_data/pics', test_df)
collator = EDPCollator()

# 3. ASİMETRİK MODELİ YÜKLE
# models.py dosyasını HuggingFace'ten çekecek şekilde hacklemiştik, 
# bu yüzden parametre göndermemize gerek yok.
model = MMTD()

# 4. EĞİTİM ARGÜMANLARI (Modern Kaggle Uyumu)
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",      # evaluation_strategy yerine modern isim
    save_strategy="epoch",
    learning_rate=5e-4,         # Orijinal kodundaki learning rate
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.0,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    report_to="none"
)

# 5. TRAINER KURULUMU
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collator,
    compute_metrics=metrics,    # Orijinal metrics fonksiyonunu bağladık
)

print("🚀 SİSTEM ANALİZ EDİLDİ VE ATEŞLENİYOR...")
trainer.train()
trainer.save_model(output_dir)
print(f"✅ EĞİTİM TAMAM! Model {output_dir} klasöründe.")

In [ ]:
!python /kaggle/working/MMTD/main.py

In [ ]:
import pandas as pd
df = pd.read_csv('DATA/email_data/EDP.csv')

# 'label' sütununun adını kendi CSV'ne göre kontrol et (Label mı, target mı, class mı?)
# Genellikle 'label' olur.
print("📌 Benzersiz Etiketler:", df['label'].unique())

# Eğer 0 ve 1 dışında bir şey (mesela 1 ve 2) varsa, onları 0 ve 1'e çekmeliyiz:
if df['label'].min() == 1:
    print("⚠️ Etiketler 1'den başlıyor! 0'a çekiliyor...")
    # Basit bir düzeltme gerekebilir

In [ ]:
import pandas as pd
# Veriyi yükle
df = pd.read_csv('DATA/email_data/EDP.csv')

print("--- VERİ YAPISI ANALİZİ ---")
print(f"Toplam Satır: {len(df)}")
print(f"Sütun İsimleri: {df.columns.tolist()}")
print("\nİlk 2 Satır:")
print(df.head(2))

# Buradaki çıktıya bakacağız. Eğer 'label' yoksa, hangi sütun sayısal (0/1) değer içeriyor?
# Eğer isim farklıysa (örn: 'target'), alttaki kodu ona göre güncelleyeceğiz.

In [ ]:
import pandas as pd

# Veriyi yükle
df = pd.read_csv('DATA/email_data/EDP.csv')

# 1. Sütun isimlerini düzelt (labels -> label)
df = df.rename(columns={'labels': 'label'})

# 2. NaN (boş) metinleri boş string ile doldur (Tokenizer çökmemesi için kritik!)
df['texts'] = df['texts'].fillna('')

# 3. Etiketleri kontrol et (Sadece 0 ve 1 olduğundan emin olalım)
print(f"📊 Mevcut etiketler: {df['label'].unique()}")

# 4. Temizlenmiş veriyi kaydet
df.to_csv('DATA/email_data/EDP.csv', index=False)
print("✅ Veri seti (EDP.csv) temizlendi ve standardize edildi.")

In [ ]:
import torch
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD
from utils import SplitData

# Veriyi yükle ve 1. fold'u ayır
split_data = SplitData('DATA/email_data/EDP.csv', 5)
train_df, _ = split_data()

# Dataset ve Collator kurulumu
ds = EDPDataset('DATA/email_data/pics', train_df)
collator = EDPCollator()

# CPU üzerinde modeli yükle (Asimetrik yapı)
model = MMTD().cpu()
model.eval()

# Tek bir batch (2 örnek) alalım
try:
    batch = collator([ds[0], ds[1]])
    
    # Boyutları yazdırıp gözle kontrol edelim
    print(f"📏 Metin boyutu: {batch['input_ids'].shape}")   # Beklenen: [2, 512]
    print(f"🖼️ Resim boyutu: {batch['pixel_values'].shape}") # Beklenen: [2, 3, 224, 224]
    print(f"🏷️ Etiketler: {batch['labels']}")

    # Modeli test et (Forward Pass)
    with torch.no_grad():
        outputs = model(**batch)
    
    print("\n✅ TEST BAŞARILI: Boyutlar uyuşuyor, forward pass sorunsuz.")
    print(f"🎯 Çıktı (Logits) Boyutu: {outputs.logits.shape}") # Beklenen: [2, 2]
    
except Exception as e:
    print(f"\n❌ BOYUT HATASI TESPİT EDİLDİ: {str(e)}")

In [ ]:
import pandas as pd
# Veriyi oku ve standardize et
df = pd.read_csv('DATA/email_data/EDP.csv')

# labels -> label yapıyoruz (Trainer'ın en sevdiği isim)
df = df.rename(columns={'labels': 'label'}) 

# Tokenizer hatasını önlemek için boş metinleri (NaN) temiz string yap
df['texts'] = df['texts'].fillna('') 

# Temizlenmiş dosyayı kaydet
df.to_csv('DATA/email_data/EDP.csv', index=False)
print("✅ 1. ADIM TAMAM: Veri seti temizlendi ve 'label' sütunu hazır!")

In [ ]:
import os

# EDP.csv dosyasının sistemdeki tam yolunu bulalım
arama_sonucu = os.popen('find /kaggle/working -name "EDP.csv"').read().strip()

if arama_sonucu:
    print(f"✅ Dosyan tam olarak burada bulundu: {arama_sonucu}")
    csv_yolu = arama_sonucu
else:
    print("❌ Dosya hiçbir yerde bulunamadı! Sanırım zipten çıkarma işlemini tekrar yapmalısın.")

In [ ]:
import pandas as pd
import os

# Tespit ettiğin mutlak yol
csv_yolu = '/kaggle/working/MMTD/DATA/email_data/EDP.csv'

# Veriyi oku ve standardize et
df = pd.read_csv(csv_yolu)

# labels -> label yapıyoruz (Trainer'ın beklediği standart isim)
if 'labels' in df.columns:
    df = df.rename(columns={'labels': 'label'}) 

# Tokenizer hatasını önlemek için boş metinleri (NaN) boş string yap
df['texts'] = df['texts'].fillna('') 

# Üzerine mutlak yolla kaydet
df.to_csv(csv_yolu, index=False)
print(f"✅ 1. ADIM TAMAM: {csv_yolu} temizlendi ve hazır!")

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
from torch import nn
from transformers import BertModel, BeitModel

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = BeitModel.from_pretrained("microsoft/beit-base-patch16-224-pt22k")
        
        # ASİMETRİK: Metni dondur, resmi eğit
        for param in self.text_encoder.parameters():
            param.requires_grad = False
            
        self.classifier = nn.Linear(768 + 768, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        # Sadece input_ids ve mask göndererek segment hatasını engelliyoruz
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_outputs.last_hidden_state[:, 0, :] # [CLS]
        
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_features = image_outputs.last_hidden_state[:, 0, :] # [CLS]
        
        combined = torch.cat((text_features, image_features), dim=1)
        logits = self.classifier(combined)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1))
            
        return type('Output', (object,), {'loss': loss, 'logits': logits})

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import torch
from transformers import TrainingArguments, Trainer
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD 
from utils import SplitData, metrics

os.environ["WANDB_MODE"] = "disabled"

# MUTLAK YOLLAR
csv_path = '/kaggle/working/MMTD/DATA/email_data/EDP.csv'
pics_path = '/kaggle/working/MMTD/DATA/email_data/pics'

# Veri ayırma
split_data = SplitData(csv_path, 5)
train_df, test_df = split_data()

train_dataset = EDPDataset(pics_path, train_df)
test_dataset = EDPDataset(pics_path, test_df)

model = MMTD()

training_args = TrainingArguments(
    output_dir='./output/PhishGuard_Asymmetric',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=EDPCollator(),
    compute_metrics=metrics,
)

print("🚀 MUTLAK YOLLAR KONTROL EDİLDİ. EĞİTİM BAŞLIYOR...")
trainer.train()

In [ ]:
!python /kaggle/working/MMTD/main.py

In [ ]:
import torch
import pandas as pd
from Email_dataset import EDPDataset, EDPCollator
from models import MMTD
from utils import SplitData

# 1. Veriyi yükle ve denetle
df = pd.read_csv('/kaggle/working/MMTD/DATA/email_data/EDP.csv')
print(f"📊 Veri Sayısı: {len(df)}")
print(f"🎯 Benzersiz Etiketler: {df['label'].unique()}") # Sadece [0, 1] görmeliyiz

# 2. Modeli CPU'da yükle
model = MMTD(num_labels=2).cpu()
collator = EDPCollator()

# 3. Veri setinden 2 örnek çek
split_data = SplitData('/kaggle/working/MMTD/DATA/email_data/EDP.csv', 5)
train_df, _ = split_data()
ds = EDPDataset('/kaggle/working/MMTD/DATA/email_data/pics', train_df)

print("\n🔍 Boyut ve Değer Denetimi Başlıyor...")
try:
    # İlk 2 örneği alıp bir batch yapalım
    batch = collator([ds[0], ds[1]])
    
    # Detaylı rapor
    print(f"--- Input IDS Max: {batch['input_ids'].max()}") # 30522'den küçük olmalı
    print(f"--- Input IDS Min: {batch['input_ids'].min()}") # 0'dan büyük/eşit olmalı
    print(f"--- Label Değerleri: {batch['labels']}")       # Sadece 0 veya 1 olmalı
    print(f"--- Image Shape: {batch['pixel_values'].shape}")
    
    # Modeli test et (Forward Pass)
    with torch.no_grad():
        outputs = model(**batch)
    
    print("\n✅ UNIT TEST BAŞARILI!")
    print(f"Logits Boyutu: {outputs.logits.shape} (Beklenen: [2, 2])")
    print(f"Loss Değeri: {outputs.loss}")

except Exception as e:
    print(f"\n❌ KRİTİK HATA TESPİT EDİLDİ: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
%%writefile -a models.py

# --- BİZİM EKLEDİĞİMİZ YENİ MODÜLLER ---
import torch.nn as nn

class AsymmetricGatedBEiTMMTD(nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(AsymmetricGatedBEiTMMTD, self).__init__()
        
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        self.hidden_size = 768
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.Sigmoid()
        )
        
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)
        self.layer_norm2 = nn.LayerNorm(self.hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size * 4, self.hidden_size)
        )
        self.dropout = nn.Dropout(0.1)

        self.pooler = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Tanh()
        )
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None):
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_last_hidden_state = text_outputs.hidden_states[12]
        image_last_hidden_state = image_outputs.hidden_states[12]
        
        attn_output, _ = self.cross_attention(
            query=text_last_hidden_state, 
            key=image_last_hidden_state, 
            value=image_last_hidden_state
        )
        
        concat_features = torch.cat([text_last_hidden_state, attn_output], dim=-1)
        gate = self.gate_network(concat_features)
        
        gated_output = text_last_hidden_state + (gate * attn_output)
        
        x = self.layer_norm1(gated_output)
        ffn_output = self.ffn(x)
        fused_state = self.layer_norm2(x + self.dropout(ffn_output))
        
        pooled_output = self.pooler(fused_state[:, 0, :])
        logits = self.classifier(self.dropout(pooled_output))
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=None, attentions=None)

In [ ]:
%%writefile main.py
from Email_dataset import EDPDataset, EDPCollator
from models import AsymmetricGatedBEiTMMTD 
from transformers import Trainer, TrainingArguments
from torch.optim import AdamW, lr_scheduler
from utils import metrics, SplitData, save_config, EvalMetrics
import wandb
import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
fold = 5
split_data = SplitData('DATA/email_data/EDP.csv', fold)

bert_checkpoint_path = './output/bert/checkpoints'
bert_folds = os.listdir(bert_checkpoint_path)
bert_checkpoints = list()
for _ in os.listdir(bert_checkpoint_path):
    bert_checkpoints.extend(os.listdir(os.path.join(bert_checkpoint_path, _)))

dit_checkpoint_path = './output/dit/checkpoints'
dit_folds = os.listdir(dit_checkpoint_path)
dit_checkpoints = list()
for _ in os.listdir(dit_checkpoint_path):
    dit_checkpoints.extend(os.listdir(os.path.join(dit_checkpoint_path, _)))

if __name__ == '__main__':
    for i in range(fold):
        wandb.init(project='Asymmetric-Gated-MMTD')
        wandb.run.name = 'AsymGated-fold-' + str(i + 1)
        train_df, test_df = split_data()
        train_dataset = EDPDataset('DATA/email_data/pics', train_df)
        test_dataset = EDPDataset('DATA/email_data/pics', test_df)

        bert_checkpoint = os.path.join(bert_checkpoint_path, bert_folds[i], bert_checkpoints[i])
        dit_checkpoint = os.path.join(dit_checkpoint_path, dit_folds[i], dit_checkpoints[i])
        
        # YENİ MODELİMİZİ BAŞLATIYORUZ
        model = AsymmetricGatedBEiTMMTD(bert_pretrain_weight=bert_checkpoint, beit_pretrain_weight=dit_checkpoint)

        # Temel modelleri donduruyoruz (Sadece yeni eklediğimiz katmanlar eğitilecek)
        for p in model.text_encoder.parameters():
            p.requires_grad = False
        for p in model.image_encoder.parameters():
            p.requires_grad = False

        filtered_parameters = []
        for p in filter(lambda p: p.requires_grad, model.parameters()):
            filtered_parameters.append(p)

        optimizer = AdamW(filtered_parameters, lr=5e-4)

        args = TrainingArguments(
            output_dir='./output/AsymMMTD/checkpoints/fold' + str(i + 1),
            logging_dir='./output/AsymMMTD/log',
            logging_strategy='epoch',
            learning_rate=5e-4,
            per_device_train_batch_size=40,
            per_device_eval_batch_size=40,
            num_train_epochs=1, # DİKKAT: Bellek (OOM) testi için 1 yapıldı. Sorun çıkmazsa 3 yapabilirsin.
            weight_decay=0.0,
            save_strategy="epoch",
            evaluation_strategy="epoch",
            load_best_model_at_end=True,
            dataloader_num_workers=0,
            dataloader_pin_memory=True,
            run_name=wandb.run.name,
            auto_find_batch_size=False,
            overwrite_output_dir=True,
            save_total_limit=5,
            remove_unused_columns=False,
            report_to=["wandb"],
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            optimizers=(optimizer, None),
            data_collator=EDPCollator(),
            compute_metrics=metrics,
        )

        # Eğitimi başlatıyoruz (Çift yazım hatası düzeltildi)
        trainer.train()

        train_acc = trainer.evaluate(eval_dataset=train_dataset)
        train_result = {'train_acc': train_acc['eval_acc'], 'train_loss': train_acc['eval_loss']}
        wandb.log(train_result)

        trainer.compute_metrics = EvalMetrics('./output/AsymMMTD/results', wandb.run.name, True)
        test_acc = trainer.evaluate(eval_dataset=test_dataset)
        test_result = {'test_acc': test_acc['eval_acc'], 'test_loss': test_acc['eval_loss']}
        wandb.log(test_result)

        wandb.config = args.to_dict()
        save_config(args.to_dict(), os.path.join('./output/AsymMMTD/configs', wandb.run.name + '.yaml'))
        wandb.finish()

In [ ]:
!python main.py

In [ ]:
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' Email_dataset.py

In [ ]:
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' Email_dataset.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' Email_dataset.py

In [ ]:
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' utils.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' utils.py
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' utils.py

In [ ]:
# utils.py dosyasını okuyoruz
with open('utils.py', 'r') as f:
    content = f.read()

# GloVe importunu zararsız bir boş sınıfla (mock) değiştiriyoruz
mock_glove = """
class GloVe:
    def __init__(self, *args, **kwargs):
        pass
"""
content = content.replace('from torchtext.vocab import GloVe', mock_glove)

# Dosyayı güncellenmiş haliyle geri yazıyoruz
with open('utils.py', 'w') as f:
    f.write(content)

print("✅ utils.py başarıyla yamalandı! torchtext belasından kurtulduk.")

In [ ]:
import re

# utils.py dosyasını okuyoruz
with open('utils.py', 'r') as f:
    content = f.read()

# İçinde 'torchtext' geçen tüm satırları siliyoruz
content = re.sub(r'^.*torchtext.*$', '', content, flags=re.MULTILINE)

# Eski kodların (bizim kullanmadığımız ama dosyada duran LSTM vb.) hata vermemesi için sahte fonksiyonlar
mocks = """
# --- MOCK TORCHTEXT KÖPRÜSÜ ---
class GloVe:
    def __init__(self, *args, **kwargs): pass
def get_tokenizer(*args, **kwargs): return lambda x: x
def build_vocab_from_iterator(*args, **kwargs): pass
# ------------------------------
"""

# Dosyayı temizlenmiş ve yamalanmış haliyle geri yazıyoruz
with open('utils.py', 'w') as f:
    f.write(mocks + "\n" + content)

print("✅ utils.py KAPSAMLI olarak temizlendi! Tüm torchtext bağları koparıldı.")

In [ ]:
!python main.py

In [ ]:
import re

# Email_dataset.py dosyasını okuyoruz
with open('Email_dataset.py', 'r') as f:
    content = f.read()

# İçinde 'torchtext' geçen tüm satırları bulup siliyoruz
content = re.sub(r'^.*torchtext.*$', '', content, flags=re.MULTILINE)

# Eski kodların çökmesini engellemek için sahte sınıfları en başa ekliyoruz
mocks = """
# --- MOCK TORCHTEXT KÖPRÜSÜ ---
class GloVe:
    def __init__(self, *args, **kwargs): pass
def get_tokenizer(*args, **kwargs): return lambda x: x
# ------------------------------
"""

# Dosyayı tertemiz haliyle geri yazıyoruz
with open('Email_dataset.py', 'w') as f:
    f.write(mocks + "\n" + content)

print("✅ Email_dataset.py de başarıyla temizlendi! Artık karşımıza çıkamaz.")

In [ ]:
# Eğer MMTD klasöründe değilsek oraya geçelim
%cd /kaggle/working/MMTD

import os
# Kodun aradığı DATA klasörünü oluşturalım
os.makedirs('DATA', exist_ok=True)

print("1. Google Drive'dan veri seti indiriliyor (Bu biraz sürebilir)...")
# gdown kullanarak linkteki ID ile zip dosyasını indiriyoruz
!gdown --id 1w4HxNf099lQ41mhMyXzbGwl429qBGvbY -O DATA/dataset.zip

print("2. Zip dosyası klasöre çıkartılıyor...")
# İnen zip dosyasını DATA klasörüne çıkartıyoruz (-q sessiz mod, ekrana çok yazı basmasın diye)
!unzip -q DATA/dataset.zip -d DATA/

print("3. Temizlik yapılıyor...")
# Yer kaplamaması için orijinal zip dosyasını siliyoruz
!rm DATA/dataset.zip

print("✅ BİTTİ! Veri seti başarıyla yüklendi ve kullanıma hazır.")

In [ ]:
!python main.py

In [ ]:
%cd /kaggle/working/MMTD/DATA


In [ ]:
!pip install -U gdown

In [ ]:
!gdown --id "1w4HxNf099lQ41mhMyXzbGwI429qBGvbY"

In [ ]:
!gdown "https://drive.google.com/uc?id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY"

In [ ]:
import torch
import torch.nn as nn
from transformers import BertConfig, BeitConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput

# models.py dosyasındaki sınıfımızı import ediyoruz
# Eğer sınıf ismini farklı kaydettiysen burayı güncelle
try:
    from models import AsymmetricGatedBEiTMMTD as PhishGuard
except ImportError:
    # Eğer models.py henüz dosyaya yazılmadıysa veya yol hatası varsa
    # Doğrudan mevcut notebook içindeki sınıf ismini kullanabilirsin
    print("Models.py bulunamadı, lütfen sınıfın tanımlı olduğu hücreyi çalıştırdığınızdan emin olun.")

def run_sanity_check():
    print("🚀 Asimetrik & Dinamik Gating Testi Başlatılıyor...")
    
    # 1. Donanım Hazırlığı
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"💻 Kullanılan Donanım: {device}")

    # 2. Model Kurulumu (Hafif konfigürasyon ile test ediyoruz)
    # Not: Gerçek ağırlıklar yüklü olmasa bile mimariyi test edebiliriz
    try:
        model = PhishGuard().to(device)
        model.eval() # Test modu
        print("✅ Model başarıyla ayağa kaldırıldı.")
    except Exception as e:
        print(f"❌ Model kurulum hatası: {e}")
        return

    # 3. Sahte (Dummy) Veri Üretimi
    batch_size = 2 # 2 adet örnek e-posta
    seq_length = 128 # 128 kelimelik metin
    
    # Metin Girişleri (BERT formatında)
    dummy_input_ids = torch.randint(0, 30000, (batch_size, seq_length)).to(device)
    dummy_attention_mask = torch.ones(batch_size, seq_length).to(device)
    dummy_token_type_ids = torch.zeros(batch_size, seq_length, dtype=torch.long).to(device)
    
    # Görsel Girişleri (BEiT/DiT formatında: 3 kanal, 224x224 piksel)
    dummy_pixel_values = torch.randn(batch_size, 3, 224, 224).to(device)

    print("4. İleri Yönlü Geçiş (Forward Pass) Yapılıyor...")
    
    # 4. Test İşlemi
    try:
        with torch.no_grad():
            outputs = model(
                input_ids=dummy_input_ids,
                token_type_ids=dummy_token_type_ids,
                attention_mask=dummy_attention_mask,
                pixel_values=dummy_pixel_values
            )
        
        # Sonuçların Analizi
        logits = outputs.logits
        print("\n" + "="*30)
        print("🎊 TEST BAŞARIYLA TAMAMLANDI! 🎊")
        print("="*30)
        print(f"Çıktı (Logits) Boyutu: {logits.shape}")
        print(f"Beklenen Boyut: [{batch_size}, 2]")
        
        if logits.shape == (batch_size, 2):
            print("\n✔️ Matris boyutları uyumlu.")
            print("✔️ Asimetrik Cross-Attention başarıyla hesaplandı.")
            print("✔️ Dinamik Gating (Kapı) mekanizması tensörleri süzdü.")
            print("✔️ Model GPU/CPU belleğine sorunsuz sığdı.")
        
    except Exception as e:
        print("\n❌ MATRİS ÇARPIŞMASI VEYA MİMARİ HATASI:")
        print(e)

# Testi çalıştır
if __name__ == "__main__":
    run_sanity_check()

In [ ]:
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BeitForImageClassification, BeitConfig, BertConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput

# ==========================================
# 1. MODEL TANIMI (Doğrudan Hücre İçinde)
# ==========================================
class AsymmetricGatedBEiTMMTD(nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(AsymmetricGatedBEiTMMTD, self).__init__()
        
        # Enkoderlar
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        self.hidden_size = 768
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

        # Asymmetric Cross-Attention (Metin Sorgular, Görsel Cevap Verir)
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # Dynamic Gating (Kapı Mekanizması)
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.Sigmoid()
        )
        
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)
        self.layer_norm2 = nn.LayerNorm(self.hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size * 4, self.hidden_size)
        )
        self.dropout = nn.Dropout(0.1)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None):
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        # 12. katman çıktılarını al (Last Hidden State)
        text_last_hidden_state = text_outputs.hidden_states[12]
        image_last_hidden_state = image_outputs.hidden_states[12]
        
        # Asymmetric Cross Attention
        attn_output, _ = self.cross_attention(
            query=text_last_hidden_state, 
            key=image_last_hidden_state, 
            value=image_last_hidden_state
        )
        
        # Dynamic Gating Logic
        concat_features = torch.cat([text_last_hidden_state, attn_output], dim=-1)
        gate = self.gate_network(concat_features)
        gated_output = text_last_hidden_state + (gate * attn_output)
        
        # Residual & FFN
        x = self.layer_norm1(gated_output)
        fused_state = self.layer_norm2(x + self.dropout(self.ffn(x)))
        
        # Pool & Classify
        pooled_output = self.pooler(fused_state[:, 0, :])
        logits = self.classifier(self.dropout(pooled_output))
        
        return SequenceClassifierOutput(logits=logits)

# ==========================================
# 2. TEST DÖNGÜSÜ (Sanity Check)
# ==========================================
print("🚀 Asimetrik & Dinamik Gating Çarpışma Testi Başlıyor...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Donanım: {device}")

try:
    # Modeli Başlat (Ağırlıksız, sadece mimari test)
    test_model = AsymmetricGatedBEiTMMTD().to(device)
    test_model.eval()

    # Sahte Veriler
    b, s = 2, 128 # Batch size 2, Seq length 128
    d_ids = torch.randint(0, 2000, (b, s)).to(device)
    d_mask = torch.ones(b, s).to(device)
    d_type = torch.zeros(b, s, dtype=torch.long).to(device)
    d_pix = torch.randn(b, 3, 224, 224).to(device)

    with torch.no_grad():
        out = test_model(d_ids, d_type, d_mask, d_pix)

    print("\n" + "="*35)
    print("🎊 MİMARİ DOĞRULANDI! 🎊")
    print(f"Logit Boyutu: {out.logits.shape} (Beklenen: [2, 2])")
    print("="*35)
    print("✔️ Asimetrik Çapraz Dikkat (Cross-Attention) Aktif.")
    print("✔️ Dinamik Geçit (Dynamic Gating) Tensörleri Süzüyor.")
    print("✔️ Model Eğitime Hazır!")

except Exception as e:
    print(f"\n❌ TEST BAŞARISIZ: {e}")

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BeitForImageClassification, BeitConfig, BertConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class AsymmetricGatedBEiTMMTD(nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(AsymmetricGatedBEiTMMTD, self).__init__()
        
        # Enkoderlar
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        self.hidden_size = 768
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

        # Asymmetric Cross-Attention
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # Dynamic Gating
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.Sigmoid()
        )
        
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)
        self.layer_norm2 = nn.LayerNorm(self.hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size * 4, self.hidden_size)
        )
        self.dropout = nn.Dropout(0.1)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1):
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_last_hidden_state = text_outputs.hidden_states[12]
        image_last_hidden_state = image_outputs.hidden_states[12]

        # --- MODALITY DROPOUT ---
        # Eğitim sırasında modalitelerden birini rastgele susturarak modelin tek taraflı bağımlılığını kırıyoruz
        if self.training and modality_dropout_prob > 0:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob: # Sadece metni tut, görseli sıfırla
                image_last_hidden_state = torch.zeros_like(image_last_hidden_state)
            elif rand_val < 2 * modality_dropout_prob: # Sadece görseli tut, metni sıfırla
                text_last_hidden_state = torch.zeros_like(text_last_hidden_state)
        # ------------------------
        
        # Asymmetric Cross Attention
        attn_output, _ = self.cross_attention(
            query=text_last_hidden_state, 
            key=image_last_hidden_state, 
            value=image_last_hidden_state
        )
        
        # Dynamic Gating
        concat_features = torch.cat([text_last_hidden_state, attn_output], dim=-1)
        gate = self.gate_network(concat_features)
        gated_output = text_last_hidden_state + (gate * attn_output)
        
        # Transformer Block (Norm + FFN)
        x = self.layer_norm1(gated_output)
        fused_state = self.layer_norm2(x + self.dropout(self.ffn(x)))
        
        # Classification
        pooled_output = self.pooler(fused_state[:, 0, :])
        logits = self.classifier(self.dropout(pooled_output))
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
!gdown --id "1w4HxNf099lQ41mhMyXzbGwI429qBGvbY"

In [ ]:
!gdown "1w4HxNf099lQ41mhMyXzbGwl429qBGvbY" -O DATA/dataset.zip

In [ ]:
!curl -L -c cookies.txt 'https://drive.google.com/uc?export=download&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY' | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1/p' > confirm.txt
!curl -L -b cookies.txt "https://drive.google.com/uc?export=download&confirm=$(cat confirm.txt)&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY" -o DATA/dataset.zip

In [ ]:
cd /kaggle/working/MMTD

In [ ]:
# 1. Çalışma alanını temizle ve klasörleri oluştur
%cd /kaggle/working
mkdir -p MMTD/DATA
cd MMTD

# 2. Google Drive'ın "virüs taraması onayı" mekanizmasını bypass ederek indir
echo "🚀 Veri seti indiriliyor, bu işlem internet hızına göre zaman alabilir..."
curl -L -c cookies.txt 'https://drive.google.com/uc?export=download&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY' | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1/p' > confirm.txt
curl -L -b cookies.txt "https://drive.google.com/uc?export=download&confirm=$(cat confirm.txt)&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY" -o DATA/dataset.zip

# 3. İndirme başarılı mı kontrol et ve zip'ten çıkar
if [ -f "DATA/dataset.zip" ]; then
    echo "✅ İndirme tamamlandı. Zip dosyası açılıyor..."
    unzip -q DATA/dataset.zip -d DATA/
    rm DATA/dataset.zip confirm.txt cookies.txt
    echo "🎊 Her şey hazır! Veri seti DATA klasörüne yerleştirildi."
else
    echo "❌ Hata: Dosya indirilemedi. Google Drive erişimi reddetmiş olabilir."
fi

In [ ]:
%%bash
# 1. Klasör yapısını garantiye al
mkdir -p /kaggle/working/MMTD/DATA
cd /kaggle/working/MMTD

echo "🌐 Google Drive bağlantısı kuruluyor..."

# 2. Google Drive'ın virüs taraması onayını simüle et
curl -L -c cookies.txt 'https://drive.google.com/uc?export=download&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY' | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1/p' > confirm.txt

# 3. Onay koduyla beraber asıl dosyayı indir
curl -L -b cookies.txt "https://drive.google.com/uc?export=download&confirm=$(cat confirm.txt)&id=1w4HxNf099lQ41mhMyXzbGwl429qBGvbY" -o DATA/dataset.zip

# 4. İndirme başarılı mı kontrol et ve aç
if [ -s "DATA/dataset.zip" ]; then
    echo "✅ İndirme tamamlandı. Dosyalar çıkartılıyor..."
    unzip -q DATA/dataset.zip -d DATA/
    rm DATA/dataset.zip confirm.txt cookies.txt
    echo "🎊 Başarılı! Veri seti MMTD/DATA klasörüne yerleştirildi."
    ls -R DATA | head -n 10  # İlk 10 dosyayı kontrol için listele
else
    echo "❌ HATA: Google Drive kotası dolmuş veya erişim engellenmiş."
    echo "Bu durumda tek yol dosyayı bilgisayarına indirip manuel yüklemek."
fi

In [ ]:
import requests
import os

def download_file_from_google_drive(id, destination):
    URL = "https://docs.google.com/uc?export=download"
    session = requests.Session()
    response = session.get(URL, params={'id': id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break
    if token:
        params = {'id': id, 'confirm': token}
        response = session.get(URL, params=params, stream=True)
    
    CHUNK_SIZE = 32768
    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk: 
                f.write(chunk)
    print(f"✅ İndirme Tamamlandı: {destination}")

# Klasörü oluştur ve indir
os.makedirs('/kaggle/working/MMTD/DATA', exist_ok=True)
drive_id = '1w4HxNf099lQ41mhMyXzbGwl429qBGvbY'
dest = '/kaggle/working/MMTD/DATA/dataset.zip'

print("🚀 Alternatif indirme metodu başlatılıyor (Tarayıcı simülasyonu)...")
download_file_from_google_drive(drive_id, dest)

# Hemen arkasından açmayı dene
print("📦 Zip açılıyor...")
!unzip -q /kaggle/working/MMTD/DATA/dataset.zip -d /kaggle/working/MMTD/DATA/
print("🎊 BİTTİ! DATA klasörünü kontrol et.")

In [ ]:
!pip install -U gdown

!gdown --fuzzy "https://drive.google.com/file/d/1w4HxNf099lQ41mhMyXzbGwI429qBGvbY/view?usp=drive_link"

In [ ]:
!gdown 1w4HxNf099lQ41mhMyXzbGwI429qBGvbY

In [ ]:
!unzip /kaggle/working/MMTD/email_data.zip -d /kaggle/working/MMTD/DATA/

In [ ]:
!cat /kaggle/working/MMTD/models.py

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BeitForImageClassification, BeitConfig, BertConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class MMTD(nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(MMTD, self).__init__()
        
        # Enkoderlar
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        self.hidden_size = 768
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

        # Asymmetric Cross-Attention
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # Dynamic Gating
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.Sigmoid()
        )
        
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)
        self.layer_norm2 = nn.LayerNorm(self.hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size * 4, self.hidden_size)
        )
        self.dropout = nn.Dropout(0.1)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1):
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_last_hidden_state = text_outputs.hidden_states[12]
        image_last_hidden_state = image_outputs.hidden_states[12]

        # --- MODALITY DROPOUT ---
        # Eğitim sırasında metin veya görselden birini rastgele sıfırlayarak dayanıklılığı artırıyoruz
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob: # Görseli kapat
                image_last_hidden_state = torch.zeros_like(image_last_hidden_state)
            elif rand_val < 2 * modality_dropout_prob: # Metni kapat
                text_last_hidden_state = torch.zeros_like(text_last_hidden_state)
        # ------------------------
        
        # Asymmetric Cross Attention
        attn_output, _ = self.cross_attention(
            query=text_last_hidden_state, 
            key=image_last_hidden_state, 
            value=image_last_hidden_state
        )
        
        # Dynamic Gating
        concat_features = torch.cat([text_last_hidden_state, attn_output], dim=-1)
        gate = self.gate_network(concat_features)
        gated_output = text_last_hidden_state + (gate * attn_output)
        
        # Transformer Block
        x = self.layer_norm1(gated_output)
        fused_state = self.layer_norm2(x + self.dropout(self.ffn(x)))
        
        # Classification
        pooled_output = self.pooler(fused_state[:, 0, :])
        logits = self.classifier(self.dropout(pooled_output))
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%cd /kaggle/working/MMTD

# Eğitim ayarlarını ufak bir test için güncelliyoruz
# 1 epoch, düşük batch size (hafıza dostu), ve her 10 adımda bir raporlama
print("🚀 Asimetrik Model 1 Epochluk Test Eğitimi Başlatılıyor...")

!python main.py \
    --num_train_epochs 1 \
    --per_device_train_batch_size 8 \
    --evaluation_strategy steps \
    --eval_steps 100 \
    --logging_steps 10 \
    --save_total_limit 1 \
    --report_to none

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
from models import MMTD as AsymmetricGatedBEiTMMTD 
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics, SplitData

# Veri yollarını unzip yaptığımız yere göre ayarlıyoruz
csv_path = 'DATA/email_data/EDP.csv'
pics_path = 'DATA/email_data/pics'

fold = 5
split_data = SplitData(csv_path, fold)

if __name__ == '__main__':
    # Test için sadece 1. fold üzerinden gidelim
    train_df, test_df = split_data()
    train_dataset = EDPDataset(pics_path, train_df)
    test_dataset = EDPDataset(pics_path, test_df)

    # Modelimizi başlatıyoruz
    model = AsymmetricGatedBEiTMMTD()

    # Eğitim Argümanları
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=1,              # Sadece 1 epoch test
        per_device_train_batch_size=8,   # Hafıza dostu batch size
        per_device_eval_batch_size=8,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="no",              # Test olduğu için kaydetmiyoruz
        report_to="none"                 # WandB vb. şimdilik kapalı
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=EDPCollator(),
        compute_metrics=metrics,
    )

    print("🏁 Gerçek veriyle asimetrik eğitim başlıyor...")
    trainer.train()

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
from models import MMTD as AsymmetricGatedBEiTMMTD 
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics, SplitData

# Veri yolları
csv_path = 'DATA/email_data/EDP.csv'
pics_path = 'DATA/email_data/pics'

fold = 5
split_data = SplitData(csv_path, fold)

if __name__ == '__main__':
    train_df, test_df = split_data()
    train_dataset = EDPDataset(pics_path, train_df)
    test_dataset = EDPDataset(pics_path, test_df)

    model = AsymmetricGatedBEiTMMTD()

    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        eval_strategy="steps",        # 'evaluation_strategy' yerine 'eval_strategy' yaptık
        eval_steps=100,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=EDPCollator(),
        compute_metrics=metrics,
    )

    print("🏁 Hata giderildi. Gerçek veriyle asimetrik eğitim başlıyor...")
    trainer.train()

In [ ]:
%cd /kaggle/working/MMTD
!python main.py

In [ ]:
!cat /kaggle/working/MMTD/models.py

In [ ]:
!cat /kaggle/working/MMTD/main.py

In [ ]:
%%writefile /kaggle/working/MMTD/models.py 
import torch
import torch.nn as nn
from transformers import BertForSequenceClassification, BeitForImageClassification, BeitConfig, BertConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class MMTD(nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(MMTD, self).__init__()
        
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        self.hidden_size = 768
        self.num_labels = 2

        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        self.gate_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.Sigmoid()
        )
        
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)
        self.layer_norm2 = nn.LayerNorm(self.hidden_size)

        self.ffn = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size * 4, self.hidden_size)
        )

        self.dropout = nn.Dropout(0.1)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(
        self,
        input_ids=None,
        token_type_ids=None,
        attention_mask=None,
        pixel_values=None,
        labels=None,
        modality_dropout_prob=0.1
    ):
        # token_type_ids yoksa üret
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)

        text_outputs = self.text_encoder(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask
        )

        image_outputs = self.image_encoder(pixel_values=pixel_values)

        # 🔥 FIX: [-1] kullan
        text_last_hidden_state = text_outputs.hidden_states[-1]
        image_last_hidden_state = image_outputs.hidden_states[-1]

        # ------------------------
        # MODALITY DROPOUT
        # ------------------------
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob:
                image_last_hidden_state = torch.zeros_like(image_last_hidden_state)
            elif rand_val < 2 * modality_dropout_prob:
                text_last_hidden_state = torch.zeros_like(text_last_hidden_state)

        # ------------------------
        # CROSS ATTENTION
        # ------------------------
        attn_output, _ = self.cross_attention(
            query=text_last_hidden_state,
            key=image_last_hidden_state,
            value=image_last_hidden_state
        )

        # ------------------------
        # GATING
        # ------------------------
        concat_features = torch.cat([text_last_hidden_state, attn_output], dim=-1)
        gate = self.gate_network(concat_features)
        gated_output = text_last_hidden_state + (gate * attn_output)

        # ------------------------
        # TRANSFORMER BLOCK
        # ------------------------
        x = self.layer_norm1(gated_output)
        fused_state = self.layer_norm2(x + self.dropout(self.ffn(x)))

        pooled_output = self.pooler(fused_state[:, 0, :])
        logits = self.classifier(self.dropout(pooled_output))

        loss = None
        if labels is not None:
            # 🔥 LABEL GUARD
            labels = labels.clamp(0, 1)
            loss = CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
!cat /kaggle/working/MMTD/Email_dataset.py

In [ ]:
%%writefile /kaggle/working/MMTD/Email_dataset.py
from transformers import BertTokenizerFast, BeitImageProcessor
import pandas as pd
import torch
import os
from PIL import Image
from torch.utils.data import DataLoader

# ------------------------------
# DATASET
# ------------------------------

class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, data_df):
        super().__init__()
        self.data_path = data_path
        self.data = data_df.reset_index(drop=True)

    def __getitem__(self, item):
        text = str(self.data.iloc[item, 0])
        pic_path = os.path.join(self.data_path, self.data.iloc[item, 1])

        # 🔥 LABEL FIX (EN KRİTİK)
        label = self.data.iloc[item, 2]
        label = int(label)
        label = 1 if label > 0 else 0

        image = Image.open(pic_path).convert("RGB")

        return text, image, label

    def __len__(self):
        return len(self.data)


# ------------------------------
# COLLATOR
# ------------------------------

class EDPCollator:
    def __init__(self):
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
        self.processor = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224')

    def __call__(self, batch):
        texts, images, labels = zip(*batch)

        text_inputs = self.tokenizer(
            list(texts),
            return_tensors='pt',
            max_length=256,
            truncation=True,
            padding='max_length'
        )

        image_inputs = self.processor(
            list(images),
            return_tensors='pt'
        )

        # 🔥 LABEL FIX
        labels = torch.tensor(labels, dtype=torch.long)

        inputs = {}
        inputs.update(text_inputs)
        inputs.update(image_inputs)
        inputs["labels"] = labels

        return inputs


# ------------------------------
# TEST
# ------------------------------

if __name__ == "__main__":
    from utils import SplitData

    csv_path = 'DATA/email_data/EDP.csv'
    pics_path = 'DATA/email_data/pics'

    split = SplitData(csv_path, 5)
    train_df, _ = split()

    dataset = EDPDataset(pics_path, train_df)
    collator = EDPCollator()

    loader = DataLoader(dataset, batch_size=4, collate_fn=collator)

    batch = next(iter(loader))

    for k, v in batch.items():
        print(k, v.shape)

    print("LABELS:", batch["labels"])
    print("MIN:", batch["labels"].min().item(), "MAX:", batch["labels"].max().item())

In [ ]:
!python /kaggle/working/MMTD/Email_dataset.py

In [ ]:
cd /kaggle/working/MMTD


In [ ]:
!python - <<EOF
from Email_dataset import EDPDataset, EDPCollator
from utils import SplitData

csv_path = 'DATA/email_data/EDP.csv'
pics_path = 'DATA/email_data/pics'

split = SplitData(csv_path, 5)
train_df, _ = split()

dataset = EDPDataset(pics_path, train_df)
sample = dataset[0]

for k,v in sample.items():
    try:
        print(k, v.shape)
    except:
        print(k, v)

print("LABEL:", sample["labels"])
EOF

In [ ]:
!python main.py

In [ ]:
%cd /kaggle/working/MMTD
!python main.py

In [ ]:
import os

# --- 1. MODELS.PY GÜNCELLEME ---
with open('/kaggle/working/MMTD/models.py', 'w') as f:
    f.write('''
import torch
import torch.nn as nn
from transformers import BertModel, BeitModel, BertConfig, BeitConfig
from transformers.models.bert.modeling_bert import SequenceClassifierOutput

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        # Tokenizer ile uyumlu olması için ikisini de 'uncased' yapıyoruz
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = BeitModel.from_pretrained("microsoft/beit-base-patch16-224-pt22k")
        
        # Asimetrik: Metni dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
            
        self.hidden_size = 768
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.gate_network = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, **kwargs):
        # input_ids içindeki değerlerin modelin sözlük boyutunu (30522) aşmadığından emin olalım
        input_ids = input_ids.clamp(0, 30521)
        
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_last = text_outputs.last_hidden_state
        image_last = image_outputs.last_hidden_state

        # Cross Attention + Gating
        attn_out, _ = self.cross_attention(query=text_last, key=image_last, value=image_last)
        gate = self.gate_network(torch.cat([text_last, attn_out], dim=-1))
        fused = text_last + (gate * attn_out)
        
        pooled = self.pooler(fused[:, 0, :])
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            labels = labels.to(logits.device).long().clamp(0, 1)
            loss = nn.CrossEntropyLoss()(logits.view(-1, 2), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)
''')

# --- 2. EMAIL_DATASET.PY GÜNCELLEME ---
with open('/kaggle/working/MMTD/Email_dataset.py', 'w') as f:
    f.write('''
from transformers import BertTokenizerFast, BeitImageProcessor
import pandas as pd
import torch
import os
from PIL import Image

class EDPDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, data_df):
        self.data_path = data_path
        self.data = data_df.reset_index(drop=True)

    def __getitem__(self, item):
        try:
            text = str(self.data.iloc[item, 0])
            pic_path = os.path.join(self.data_path, str(self.data.iloc[item, 1]))
            label = 1 if int(self.data.iloc[item, 2]) > 0 else 0
            image = Image.open(pic_path).convert("RGB")
            return text, image, label
        except Exception:
            # Bozuk bir resim gelirse güvenli bir placeholder döndür
            return "empty", Image.new('RGB', (224, 224), color='white'), 0

    def __len__(self):
        return len(self.data)

class EDPCollator:
    def __init__(self):
        # Model ile %100 uyumlu tokenizer
        self.tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
        self.processor = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224')

    def __call__(self, batch):
        texts, images, labels = zip(*batch)
        t_inputs = self.tokenizer(list(texts), return_tensors='pt', max_length=128, truncation=True, padding='max_length')
        i_inputs = self.processor(list(images), return_tensors='pt')
        return {**t_inputs, **i_inputs, "labels": torch.tensor(labels, dtype=torch.long)}
''')

# --- 3. MAIN.PY GÜNCELLEME ---
with open('/kaggle/working/MMTD/main.py', 'w') as f:
    f.write('''
import os
import pandas as pd
from models import MMTD
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics

if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv')
    train_df = df.sample(frac=0.8, random_state=42)
    test_df = df.drop(train_df.index)

    model = MMTD()
    
    args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=1,
        per_device_train_batch_size=4, # Hafıza koruması için 4
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=10,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics
    )
    trainer.train()
''')

print("✅ Tüm dosyalar 'bert-base-uncased' standardına göre harmonize edildi.")
%cd /kaggle/working/MMTD
!python main.py

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import os

# Log dosyasının yolu (main.py'deki output_dir'e göre)
log_path = '/kaggle/working/MMTD/results/trainer_state.json'

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        data = json.load(f)
    
    history = pd.DataFrame(data['log_history'])

    # Grafikleri yan yana çizelim
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss Grafiği
    ax1.plot(history['step'], history['loss'], label='Eğitim Kaybı (Loss)', color='blue')
    if 'eval_loss' in history.columns:
        eval_data = history.dropna(subset=['eval_loss'])
        ax1.plot(eval_data['step'], eval_data['eval_loss'], label='Doğrulama Kaybı (Eval Loss)', marker='o', color='orange')
    ax1.set_title('Loss Eğrisi')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Accuracy Grafiği
    if 'eval_acc' in history.columns:
        eval_acc_data = history.dropna(subset=['eval_acc'])
        ax2.plot(eval_acc_data['step'], eval_acc_data['eval_acc'], label='Doğrulama Doğruluğu (Eval Acc)', marker='s', color='green')
        ax2.set_title('Accuracy Gelişimi')
        ax2.set_xlabel('Step')
        ax2.set_ylabel('Accuracy')
        ax2.legend()

    plt.tight_layout()
    plt.savefig('/kaggle/working/egitim_grafikleri.png')
    plt.show()
else:
    print("Log dosyası bulunamadı. Lütfen yolu kontrol et.")

In [ ]:
!find /kaggle/working -name "trainer_state.json"

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import glob

# 1. DOSYAYI OTOMATİK BULALIM
# Tüm alt klasörlerde 'trainer_state.json' arar
json_files = glob.glob('/kaggle/working/**/trainer_state.json', recursive=True)

if not json_files:
    print("❌ HATA: Log dosyası hala bulunamadı. Şu anki klasöründeki dosyalar şunlar:")
    print(os.listdir('/kaggle/working'))
else:
    log_path = json_files[0]
    print(f"✅ Log dosyası bulundu: {log_path}")

    # 2. VERİYİ OKUYALIM
    with open(log_path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['log_history'])

    # 3. GRAFİKLERİ ÇİZDİRELİM
    plt.figure(figsize=(14, 5))

    # Loss Grafiği
    plt.subplot(1, 2, 1)
    if 'loss' in df.columns:
        plt.plot(df['step'], df['loss'].fillna(method='ffill'), label='Train Loss', color='blue')
    if 'eval_loss' in df.columns:
        eval_df = df.dropna(subset=['eval_loss'])
        plt.plot(eval_df['step'], eval_df['eval_loss'], label='Eval Loss', marker='o', color='red')
    plt.title('Loss Eğrisi (Asimetrik Mimari)')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.legend()

    # Accuracy Grafiği
    plt.subplot(1, 2, 2)
    if 'eval_acc' in df.columns:
        eval_acc_df = df.dropna(subset=['eval_acc'])
        plt.plot(eval_acc_df['step'], eval_acc_df['eval_acc'], label='Eval Accuracy', marker='s', color='green')
        plt.title('Accuracy Gelişimi (%88 Hedefi)')
        plt.xlabel('Step')
        plt.ylabel('Accuracy')
        plt.legend()

    plt.tight_layout()
    plt.savefig('/kaggle/working/final_grafik.png')
    plt.show()

In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

# 1. MMTD İÇİNDEKİ HER YERE BAKALIM
search_path = '/kaggle/working/MMTD/**/trainer_state.json'
json_files = glob.glob(search_path, recursive=True)

if not json_files:
    print("❌ HATA: Log dosyası hala yok. MMTD klasörünün içeriğine bakalım:")
    # İçeride ne var görelim ki yolu manuel bulalım
    for root, dirs, files in os.walk('/kaggle/working/MMTD'):
        if 'trainer_state.json' in files:
            print(f"🎯 BULDUM! Buradaymış: {os.path.join(root, 'trainer_state.json')}")
            json_files.append(os.path.join(root, 'trainer_state.json'))
            break

if json_files:
    log_path = json_files[0]
    with open(log_path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['log_history'])
    
    # GRAFİK ÇİZİMİ
    plt.figure(figsize=(15, 6))
    
    # Loss
    plt.subplot(1, 2, 1)
    if 'loss' in df.columns:
        plt.plot(df['step'], df['loss'].ffill(), label='Train Loss', alpha=0.5)
    if 'eval_loss' in df.columns:
        eval_df = df.dropna(subset=['eval_loss'])
        plt.plot(eval_df['step'], eval_df['eval_loss'], 'ro-', label='Eval Loss')
    plt.title('Asimetrik Mimari Loss (Kayıp) Eğrisi')
    plt.legend()

    # Accuracy
    plt.subplot(1, 2, 2)
    if 'eval_acc' in df.columns:
        eval_acc_df = df.dropna(subset=['eval_acc'])
        plt.plot(eval_acc_df['step'], eval_acc_df['eval_acc'], 'gs-', label='Eval Accuracy')
        plt.axhline(y=0.8818, color='r', linestyle='--', label='Target 88%')
    plt.title('Accuracy (Doğruluk) Gelişimi')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("❓ Garip... MMTD klasöründe de bulamadım. main.py içinde output_dir tam olarak neydi?")

In [ ]:
!ls -R /kaggle/working/MMTD/results

In [ ]:
import os

# Tüm Kaggle klasörünü tara
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file == 'trainer_state.json':
            print(f"🎯 BULDUM! Dosya tam olarak burada: {os.path.join(root, file)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Eğer 'trainer' objesi hala duruyorsa logları buradan çekebiliriz
try:
    history = pd.DataFrame(trainer.state.log_history)
    print("✅ Loglar trainer objesinden başarıyla çekildi!")
    
    # Grafik çizdirme
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['step'], history['loss'].ffill(), label='Train Loss')
    if 'eval_loss' in history.columns:
        plt.plot(history['step'], history['eval_loss'].dropna(), 'ro-', label='Eval Loss')
    plt.legend(); plt.title("Loss")

    plt.subplot(1, 2, 2)
    if 'eval_acc' in history.columns:
        plt.plot(history['step'], history['eval_acc'].dropna(), 'gs-', label='Accuracy')
    plt.legend(); plt.title("Accuracy")
    plt.show()
    
except NameError:
    print("❌ 'trainer' objesi hafızada bulunamadı. Muhtemelen kernel kapandı veya başka bir sorun var.")

In [ ]:
!cat /kaggle/working/MMTD/main.py

In [ ]:
!cat /kaggle/working/MMTD/models.py

In [ ]:
!cat /kaggle/working/MMTD/Email_dataset.py

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        # Asimetrik Yapı: Metin için BERT, Belge Görselleri için DiT (ViT tabanlı)
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")
        
        # Asimetrik: Metin tarafını donduruyoruz (Frozen BERT)
        for param in self.text_encoder.parameters():
            param.requires_grad = False
            
        self.hidden_size = 768
        self.cross_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.gate_network = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1, **kwargs):
        # input_ids içindeki değerlerin modelin sözlük boyutunu (30522) aşmadığından emin olalım
        input_ids = input_ids.clamp(0, 30521)
        
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_last = text_outputs.last_hidden_state
        image_last = image_outputs.last_hidden_state

        # 🔥 MODALITY DROPOUT (Gürültü Direnci - Robustness)
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob: 
                # %10 ihtimalle resmi sıfırla (sadece metne güven)
                image_last = torch.zeros_like(image_last)
            elif rand_val < 2 * modality_dropout_prob: 
                # %10 ihtimalle metni sıfırla (sadece resme güven)
                text_last = torch.zeros_like(text_last)

        # Cross Attention + Dinamik Gating
        attn_out, _ = self.cross_attention(query=text_last, key=image_last, value=image_last)
        gate = self.gate_network(torch.cat([text_last, attn_out], dim=-1))
        fused = text_last + (gate * attn_out)
        
        pooled = self.pooler(fused[:, 0, :])
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            labels = labels.to(logits.device).long().clamp(0, 1)
            loss = nn.CrossEntropyLoss()(logits.view(-1, 2), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics

if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv')
    train_df = df.sample(frac=0.8, random_state=42)
    test_df = df.drop(train_df.index)

    model = MMTD()
    
    args = TrainingArguments(
        output_dir='/kaggle/working/MMTD/results', # Dosya yolunu sabitledik
        num_train_epochs=3, # Modality Dropout'un öğrenmesi için 3 epoch idealdir
        per_device_train_batch_size=4, 
        eval_strategy="epoch", # Her epoch bitiminde eval yap (başarıyı net gör)
        save_strategy="epoch", # Her epoch bitiminde ağırlıkları diske yaz
        logging_steps=50,
        save_total_limit=2, # Disk dolmasın diye sadece en iyi/son 2 modeli sakla
        load_best_model_at_end=True, # Eğitim bitince otomatik olarak en iyi skoru alanı seç
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics
    )
    
    print("🚀 DiT ve Modality Dropout ile asimetrik eğitim başlıyor...")
    trainer.train()
    
    # KTÜ Sunumu ve testler için garanti kayıt noktası
    trainer.save_model('/kaggle/working/MMTD_Final_Model')
    print("✅ Eğitim bitti! Model '/kaggle/working/MMTD_Final_Model' konumunda güvende.")

In [ ]:
%%writefile /kaggle/working/MMTD/models.py
import torch
import torch.nn as nn
from transformers import BertModel, ViTModel
from transformers.models.bert.modeling_bert import SequenceClassifierOutput
from torch.nn import CrossEntropyLoss

class MMTD(nn.Module):
    def __init__(self, num_labels=2):
        super(MMTD, self).__init__()
        
        # Enkoderlar: BERT ve DiT (Document Image Transformer)
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("microsoft/dit-base")
        
        # 🔥 ASİMETRİK YAPI: Metni Dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        
        self.hidden_size = 768
        
        # 🔥 BIDIRECTIONAL CROSS-ATTENTION
        # 1. Metin resme bakar (Text-to-Image)
        self.t2i_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        # 2. Resim metne bakar (Image-to-Text)
        self.i2t_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # 🔥 DİNAMİK GATING (Her iki yön için)
        self.gate_t2i = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        self.gate_i2t = nn.Sequential(nn.Linear(self.hidden_size * 2, self.hidden_size), nn.Sigmoid())
        
        self.layer_norm = nn.LayerNorm(self.hidden_size)
        self.pooler = nn.Sequential(nn.Linear(self.hidden_size, self.hidden_size), nn.Tanh())
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, modality_dropout_prob=0.1, **kwargs):
        input_ids = input_ids.clamp(0, 30521)
        
        text_last = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        image_last = self.image_encoder(pixel_values=pixel_values).last_hidden_state

        # 🔥 MODALITY DROPOUT (Gürültü Direnci)
        if self.training:
            rand_val = torch.rand(1).item()
            if rand_val < modality_dropout_prob:
                image_last = torch.zeros_like(image_last)
            elif rand_val < 2 * modality_dropout_prob:
                text_last = torch.zeros_like(text_last)

        # 🔥 BIDIRECTIONAL FUSION
        # Text-to-Image Attention
        attn_t2i, _ = self.t2i_attention(query=text_last, key=image_last, value=image_last)
        gate_t = self.gate_t2i(torch.cat([text_last, attn_t2i], dim=-1))
        fused_text = text_last + (gate_t * attn_t2i)
        
        # Image-to-Text Attention
        attn_i2t, _ = self.i2t_attention(query=image_last, key=text_last, value=text_last)
        gate_i = self.gate_i2t(torch.cat([image_last, attn_i2t], dim=-1))
        fused_image = image_last + (gate_i * attn_i2t)

        # Final Combine (Metin yolu üzerinden pooling)
        # Resimden gelen bilgileri metin yoluna normalize ederek ekliyoruz
        combined = self.layer_norm(fused_text + fused_image[:, :fused_text.size(1), :])
        
        pooled = self.pooler(combined[:, 0, :])
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            labels = labels.clamp(0, 1).long()
            loss = CrossEntropyLoss()(logits.view(-1, 2), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [ ]:
%%writefile /kaggle/working/MMTD/main.py
import os
import pandas as pd
from models import MMTD
from Email_dataset import EDPDataset, EDPCollator
from transformers import Trainer, TrainingArguments
from utils import metrics

if __name__ == "__main__":
    df = pd.read_csv('DATA/email_data/EDP.csv')
    train_df = df.sample(frac=0.8, random_state=42)
    test_df = df.drop(train_df.index)

    model = MMTD()
    
    # 🔥 KAYIT AYARLARI
    args = TrainingArguments(
        output_dir='/kaggle/working/results',
        num_train_epochs=3, 
        per_device_train_batch_size=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_acc",
        save_total_limit=2,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=EDPDataset('DATA/email_data/pics', train_df),
        eval_dataset=EDPDataset('DATA/email_data/pics', test_df),
        data_collator=EDPCollator(),
        compute_metrics=metrics
    )
    
    print("🚀 DiT, Bidirectional Attention ve Modality Dropout ile eğitim başlıyor...")
    trainer.train()
    trainer.save_model('/kaggle/working/MMTD_Final_Model')

In [ ]:
# 1. Eğitimi Başlat
!python /kaggle/working/MMTD/main.py

# 2. Eğitim bittikten sonra grafikleri çiz (Aynı hücrede eğitim bittikten sonra çalışır)
import matplotlib.pyplot as plt
import pandas as pd
import json
import glob

# Log dosyasını bul
log_files = glob.glob('/kaggle/working/results/**/trainer_state.json', recursive=True)
if log_files:
    with open(log_files[0], 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['log_history'])
    
    plt.figure(figsize=(15, 5))
    
    # Loss Grafiği
    plt.subplot(1, 2, 1)
    plt.plot(df['step'], df['loss'].ffill(), label='Train Loss')
    if 'eval_loss' in df.columns:
        plt.plot(df.dropna(subset=['eval_loss'])['step'], df.dropna(subset=['eval_loss'])['eval_loss'], 'ro-', label='Eval Loss')
    plt.title('Loss Eğrisi'); plt.legend()

    # Accuracy Grafiği
    plt.subplot(1, 2, 2)
    if 'eval_acc' in df.columns:
        plt.plot(df.dropna(subset=['eval_acc'])['step'], df.dropna(subset=['eval_acc'])['eval_acc'], 'gs-', label='Eval Accuracy')
    plt.title('Accuracy Gelişimi'); plt.legend()
    
    plt.savefig('/kaggle/working/final_results.png')
    plt.show()
    print("✅ Grafikler kaydedildi: /kaggle/working/final_results.png")

In [ ]:
import shutil
from IPython.display import FileLink

print("📦 Dosyalar zıpleniyor, lütfen bekleyin (Model boyutu büyük olduğu için 1-2 dakika sürebilir)...")

# 1. En iyi modelini (Final Model) ziple
shutil.make_archive('/kaggle/working/Benim_Efsane_Modelim', 'zip', '/kaggle/working/MMTD_Final_Model')

# 2. Ara Checkpoint'leri ziple (Ne olur ne olmaz yedeğimiz olsun)
shutil.make_archive('/kaggle/working/Ara_Checkpointler', 'zip', '/kaggle/working/results')

print("✅ Zipleme tamamlandı! Aşağıdaki linklere tıklayarak indirebilirsin:")
display(FileLink('Benim_Efsane_Modelim.zip'))
display(FileLink('Ara_Checkpointler.zip'))
display(FileLink('final_results.png'))

In [ ]:
import os
import shutil
from IPython.display import FileLink

# 1. Kaggle'a "Tam olarak buradayız, kafan karışmasın" diyoruz
os.chdir('/kaggle/working')

print("📦 3. Epoch sonundaki %92'lik şampiyon model zıpleniyor...")

# 2. Doğrudan sağda gördüğümüz checkpoint-9366 klasörünü ziple
shutil.make_archive('KTU_Final_Model', 'zip', '/kaggle/working/results/checkpoint-9366')

print("✅ Hazır! Aşağıdaki linklere tıklayarak indirebilirsin:")
display(FileLink('KTU_Final_Model.zip'))
display(FileLink('final_results.png'))

In [ ]:
import os
import shutil
from IPython.display import FileLink

# 1. Eski (işimize yaramayan) checkpoint'i sil
eski_ckpt = '/kaggle/working/results/checkpoint-6244'
if os.path.exists(eski_ckpt):
    shutil.rmtree(eski_ckpt)
    print("🗑️ Eski checkpoint silindi (Yer açıldı).")

# 2. Şampiyon modelin içindeki devasa "optimizer" dosyasını sil (Artık eğitime devam etmeyeceğimiz için buna gerek yok)
opt_file = '/kaggle/working/results/checkpoint-9366/optimizer.pt'
if os.path.exists(opt_file):
    os.remove(opt_file)
    print("🗑️ Optimizer dosyası silindi (Ekstra yer açıldı).")

# 3. Yarıda kalan bozuk zip dosyasını sil
bozuk_zip = '/kaggle/working/KTU_Final_Model.zip'
if os.path.exists(bozuk_zip):
    os.remove(bozuk_zip)

print("✅ Temizlik tamamlandı! Disk rahatladı. Şimdi ziplemeyi tekrar deneyebiliriz.")

# 4. Şimdi SADECE temizlenmiş şampiyon modeli ziple
print("📦 Temizlenmiş model zıpleniyor...")
shutil.make_archive('/kaggle/working/KTU_Final_Model_Temiz', 'zip', '/kaggle/working/results/checkpoint-9366')

print("🎉 İşlem Tamam! Aşağıdaki linke tıklayarak indirebilirsin:")
display(FileLink('KTU_Final_Model_Temiz.zip'))
display(FileLink('final_results.png'))

In [ ]:
import os
import glob
import shutil

print("🧹 Disk temizliği başlıyor...")

# 1. Yarım kalan ve tam inen tüm zip kalıntılarını bul ve sil
zip_dosyalari = glob.glob('/kaggle/working/MMTD/email_data.zip*')
for dosya in zip_dosyalari:
    os.remove(dosya)
    print(f"🗑️ Silindi: {dosya}")

# 2. Eğer zipten yarım yamalak çıkan DATA klasörü varsa onu da sıfırlayalım (Temiz başlangıç için)
data_klasoru = '/kaggle/working/MMTD/DATA'
if os.path.exists(data_klasoru):
    shutil.rmtree(data_klasoru)
    print(f"🗑️ Klasör silindi: {data_klasoru}")

print("✅ Disk tamamen boşaltıldı ve rahatladı!")